# 06 · 服务器结果包离线分析

把邮件中的 ZIP（或所有 `.partNNN`）和 `.parts.json` 放在同一目录。本 notebook 校验哈希、安全解包并读取现有结果；不访问数据库、不重新训练、也不自动加载模型。checkpoint 重放仍需要原始 `panel.npz`。

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/res/rl_experimental').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.res.rl_experimental.bundle import analyze_bundle  # noqa: E402
print('Python:', sys.executable)


In [ ]:
BUNDLE = Path('CHANGE_ME.parts.json')
ANALYSIS_ROOT = repo_root / 'results/rl_experimental/received'
if not BUNDLE.is_file():
    raise ValueError('请把 BUNDLE 改为邮件附件中的 .parts.json；ZIP/分卷须与它放在同一目录')
RUN_DIR = analyze_bundle(BUNDLE, ANALYSIS_ROOT)
metrics = json.loads((RUN_DIR / 'metrics.json').read_text(encoding='utf-8'))
display({key: metrics[key] for key in ['test', 'alpha_equal_weight_baseline', 'robust_top50_baseline', 'test_comparison']})
display(HTML((RUN_DIR / 'report.html').read_text(encoding='utf-8')))


In [ ]:
updates = pd.read_csv(RUN_DIR / 'update_history.csv')
validations = pd.read_csv(RUN_DIR / 'validation_updates.csv')
reward_columns = [column for column in updates if column.startswith('rollout/reward_component/')]
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(updates.environment_steps, updates['train/value_loss'], marker='.')
axes[0].set_ylabel('Value loss')
axes[1].plot(validations.environment_steps, validations['validation/ppo/final_nav'], marker='.', label='PPO')
axes[1].plot(validations.environment_steps, validations['validation/equal_weight/final_nav'], marker='.', label='Equal weight')
axes[1].set_ylabel('Validation NAV')
axes[1].legend()
for column in reward_columns:
    axes[2].plot(updates.environment_steps, updates[column], marker='.', label=column.split('/')[-1])
axes[2].set_ylabel('Reward contribution')
axes[2].set_xlabel('Environment steps')
if reward_columns:
    axes[2].legend()
for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()


## TensorBoard

下面只读取包中的事件文件。若内嵌页面不可用，可在终端运行 `.venv/bin/python -m tensorboard.main --logdir <RUN_DIR>/tensorboard --host localhost --port 6006`。

In [ ]:
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic('load_ext', 'tensorboard')
    ipython.run_line_magic('tensorboard', f'--logdir "{RUN_DIR / "tensorboard"}" --host localhost --port 6006')
